# Retail Customer Analytics & RFM Segmentation 🛒📊

## Business Problem
An online retail store wants to understand its customer base better to create targeted marketing campaigns. Treating all customers the same leads to inefficient marketing spend and lower conversion rates.

**Goals:**
- **Clean and preprocess** transactional retail data.
- **Calculate Recency, Frequency, and Monetary (RFM)** metrics for every customer.
- **Segment customers** into actionable groups (e.g., Champions, At-Risk, Loyal Customers) to drive personalized marketing strategies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

## 1. Data Loading & Initial Inspection

In [ ]:
# Load the dataset
df = pd.read_csv('retail.csv', encoding='latin-1')
display(df.head())

print("\nDataset Info:")
df.info()

## 2. Data Cleaning & Preprocessing

In [ ]:
# 1. Handle Missing Values
print(f"Missing values before:\n{df.isnull().sum()}\n")
df = df.dropna(subset=["CustomerID"])

# 2. Remove Duplicates
df = df.drop_duplicates()

# 3. Remove Cancelled Orders (Quantity < 0)
df = df[df["Quantity"] > 0]

# 4. Calculate Revenue
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# 5. Convert InvoiceDate to datetime object
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("Data cleaning complete. Data shape:", df.shape)

## 3. High-Level Metrics & Exploratory Data Analysis (EDA)

In [ ]:
print(f"Total Revenue: ${df['Revenue'].sum():,.2f}")
print(f"Total Unique Customers: {df['CustomerID'].nunique():,}")
print(f"Total Invoices: {df['InvoiceNo'].nunique():,}")

# Top 10 Customers by Revenue
top_customers = df.groupby("CustomerID")["Revenue"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_customers.index.astype(str), y=top_customers.values, palette="viridis")
plt.title("Top 10 Customers by Revenue", fontsize=14)
plt.xlabel("Customer ID")
plt.ylabel("Total Revenue ($)")
plt.show()

## 4. RFM (Recency, Frequency, Monetary) Analysis
To segment customers, we need to calculate three metrics for each customer:
- **Recency:** Days since their last purchase.
- **Frequency:** Number of purchases.
- **Monetary:** Total money spent.

In [ ]:
# Set snapshot date as 1 day after the last transaction in the dataset
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# Aggregate data on a customer level
rfm = df.groupby(['CustomerID']).agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'Revenue': 'sum'                                         # Monetary
}).reset_index()

# Rename columns
rfm.rename(columns={'InvoiceDate': 'Recency', 
                    'InvoiceNo': 'Frequency', 
                    'Revenue': 'MonetaryValue'}, inplace=True)

display(rfm.head())

### Creating RFM Scores
We divide the metrics into quartiles (1-4) to assign a score. 
- For Recency, a *lower* number is better (4 is best).
- For Frequency and Monetary, a *higher* number is better (4 is best).

In [ ]:
# Calculate Quartiles
r_labels = range(4, 0, -1) # 4 is best (most recent)
f_labels = range(1, 5)     # 4 is best (highest frequency)
m_labels = range(1, 5)     # 4 is best (highest spend)

r_quartiles = pd.qcut(rfm['Recency'], q=4, labels=r_labels)

# Handling potential duplicate edges in Frequency (many customers might have 1 purchase)
f_quartiles = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=f_labels)
m_quartiles = pd.qcut(rfm['MonetaryValue'], q=4, labels=m_labels)

rfm = rfm.assign(R=r_quartiles.values, F=f_quartiles.values, M=m_quartiles.values)

# Create an RFM Segment string and RFM Score
def join_rfm(x): return str(int(x['R'])) + str(int(x['F'])) + str(int(x['M']))
rfm['RFM_Segment'] = rfm.apply(join_rfm, axis=1)
rfm['RFM_Score'] = rfm[['R','F','M']].sum(axis=1)

display(rfm.head())

## 5. Customer Segmentation
Based on their RFM Score, we can group customers into distinct segments.

In [ ]:
def segment_me(df):
    if df['RFM_Score'] >= 10:
        return 'Champions'
    elif (df['RFM_Score'] >= 6) and (df['RFM_Score'] < 10):
        return 'Loyal/Active'
    elif (df['RFM_Score'] >= 4) and (df['RFM_Score'] < 6):
        return 'At Risk'
    else:
        return 'Lost'

rfm['Customer_Segment'] = rfm.apply(segment_me, axis=1)

# Visualize Segments
plt.figure(figsize=(8, 5))
sns.countplot(x='Customer_Segment', data=rfm, palette='Set2', order=['Champions', 'Loyal/Active', 'At Risk', 'Lost'])
plt.title('Customer Segments Distribution', fontsize=14)
plt.ylabel('Number of Customers')
plt.show()

# Key Business Insights & Recommendations

Based on the RFM Analysis, we have successfully categorized the customer base into actionable segments. 

### Segment Profiles:
- **Champions (Score 10-12):** Bought recently, buy often, and spend the most.
- **Loyal/Active (Score 6-9):** Consistent spenders and frequent buyers.
- **At Risk (Score 4-5):** Used to buy often but haven't purchased recently.
- **Lost (Score 3):** Lowest recency, frequency, and monetary values.

### Actionable Marketing Strategies:
1. **Champions:** Reward them! Provide early access to new products, exclusive discounts, or VIP loyalty programs to maintain their high engagement.
2. **Loyal/Active:** Upsell and cross-sell. Recommend higher-value items based on their past purchases.
3. **At Risk:** Send personalized "We miss you" campaigns with aggressive win-back discounts or limited-time offers to re-engage them before they churn completely.
4. **Lost:** Minimize marketing spend here. Ignore or run low-cost automated email campaigns rather than expensive direct marketing.